# EPL Match Outcome Predictor

Predicting English Premier League match outcomes using historical data from 1993 to 2023. This project covers exploratory data analysis, SQL queries, feature engineering, classification, regression, clustering, and time series analysis across 12,026 matches spanning 31 seasons.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 2. Load and Explore

In [ ]:
df = pd.read_csv('../data/raw/premier-league-matches.csv')
df['Date'] = pd.to_datetime(df['Date'])

print('Shape:', df.shape)
print('Seasons:', df['Season_End_Year'].min(), 'to', df['Season_End_Year'].max())
print('Teams:', df['Home'].nunique())
print('Missing values:', df.isna().sum().sum())
df.head(10)

In [ ]:
# Result distribution
ftr_counts = df['FTR'].value_counts()
ftr_pct = df['FTR'].value_counts(normalize=True).round(3) * 100
result_summary = pd.DataFrame({'Count': ftr_counts, 'Percent': ftr_pct})
result_summary.index = result_summary.index.map({'H': 'Home Win', 'A': 'Away Win', 'D': 'Draw'})
print('Full-time result breakdown:')
result_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ftr_vals = df['FTR'].value_counts()[['H', 'A', 'D']].values
axes[0].pie(ftr_vals, labels=['Home Win', 'Away Win', 'Draw'], autopct='%1.1f%%',
            colors=['steelblue', 'tomato', 'gray'])
axes[0].set_title('Match outcome split (1993-2023)')

axes[1].hist(df['HomeGoals'], bins=range(0, 12), color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Goals')
axes[1].set_title('Home goals per match')

axes[2].hist(df['AwayGoals'], bins=range(0, 12), color='tomato', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Goals')
axes[2].set_title('Away goals per match')

plt.tight_layout()
plt.show()

print(f"Average home goals: {df['HomeGoals'].mean():.2f}")
print(f"Average away goals: {df['AwayGoals'].mean():.2f}")
print(f"Average total goals per match: {(df['HomeGoals'] + df['AwayGoals']).mean():.2f}")

In [ ]:
# Seasonal trends
seasonal = df.groupby('Season_End_Year').agg(
    matches=('FTR', 'count'),
    home_goals=('HomeGoals', 'sum'),
    away_goals=('AwayGoals', 'sum'),
    home_wins=('FTR', lambda x: (x == 'H').sum()),
    away_wins=('FTR', lambda x: (x == 'A').sum()),
).reset_index()
seasonal['goals_per_match'] = (seasonal['home_goals'] + seasonal['away_goals']) / seasonal['matches']
seasonal['home_win_rate'] = seasonal['home_wins'] / seasonal['matches']
seasonal['away_win_rate'] = seasonal['away_wins'] / seasonal['matches']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(seasonal['Season_End_Year'], seasonal['goals_per_match'], marker='o', linewidth=1.5, color='steelblue')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Goals per match')
axes[0].set_title('Goals per match over time')

axes[1].plot(seasonal['Season_End_Year'], seasonal['home_win_rate'], marker='o', label='Home win rate', color='steelblue')
axes[1].plot(seasonal['Season_End_Year'], seasonal['away_win_rate'], marker='o', label='Away win rate', color='tomato')
axes[1].set_xlabel('Season')
axes[1].set_ylabel('Win rate')
axes[1].set_title('Home vs away win rate over time')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 teams by all-time wins
home_wins = df[df['FTR'] == 'H'].groupby('Home').size().rename('home_wins')
away_wins = df[df['FTR'] == 'A'].groupby('Away').size().rename('away_wins')
team_wins = pd.concat([home_wins, away_wins], axis=1).fillna(0)
team_wins['total_wins'] = team_wins['home_wins'] + team_wins['away_wins']
team_wins = team_wins.sort_values('total_wins', ascending=True).tail(15)

plt.figure(figsize=(10, 6))
plt.barh(team_wins.index, team_wins['total_wins'], color='steelblue')
plt.xlabel('Total wins')
plt.title('Top 15 teams by all-time wins (1993-2023)')
plt.tight_layout()
plt.show()

## 3. SQL Analysis

I load the match data into a SQLite database and run SQL queries to answer specific questions. This is the same data as above but accessed through SQL rather than pandas.

In [ ]:
conn = sqlite3.connect('../data/processed/epl.db')
df.to_sql('matches', conn, if_exists='replace', index=False)
print('Table loaded. Row count:')
pd.read_sql('SELECT COUNT(*) AS total_matches FROM matches', conn)

In [ ]:
# Goals and matches by decade
query = '''
SELECT
    CASE
        WHEN Season_End_Year BETWEEN 1993 AND 2002 THEN '1993-2002'
        WHEN Season_End_Year BETWEEN 2003 AND 2012 THEN '2003-2012'
        ELSE '2013-2023'
    END AS decade,
    COUNT(*) AS matches,
    SUM(HomeGoals + AwayGoals) AS total_goals,
    ROUND(AVG(HomeGoals + AwayGoals), 2) AS avg_goals_per_match
FROM matches
GROUP BY decade
ORDER BY decade
'''
print('Goals by decade:')
pd.read_sql(query, conn)

In [ ]:
# Top 10 teams by total wins
query = '''
SELECT team, SUM(wins) AS total_wins
FROM (
    SELECT Home AS team, COUNT(*) AS wins FROM matches WHERE FTR = 'H' GROUP BY Home
    UNION ALL
    SELECT Away AS team, COUNT(*) AS wins FROM matches WHERE FTR = 'A' GROUP BY Away
) combined
GROUP BY team
ORDER BY total_wins DESC
LIMIT 10
'''
pd.read_sql(query, conn)

In [ ]:
# Home advantage overall
query = '''
SELECT
    ROUND(100.0 * SUM(CASE WHEN FTR = 'H' THEN 1 ELSE 0 END) / COUNT(*), 1) AS home_win_pct,
    ROUND(100.0 * SUM(CASE WHEN FTR = 'A' THEN 1 ELSE 0 END) / COUNT(*), 1) AS away_win_pct,
    ROUND(100.0 * SUM(CASE WHEN FTR = 'D' THEN 1 ELSE 0 END) / COUNT(*), 1) AS draw_pct
FROM matches
'''
pd.read_sql(query, conn)

In [ ]:
# Highest scoring seasons
query = '''
SELECT
    Season_End_Year,
    COUNT(*) AS matches,
    SUM(HomeGoals + AwayGoals) AS total_goals,
    ROUND(AVG(HomeGoals + AwayGoals), 2) AS avg_goals_per_match
FROM matches
GROUP BY Season_End_Year
ORDER BY avg_goals_per_match DESC
LIMIT 10
'''
pd.read_sql(query, conn)